# Pokemon Analysis

Load the raw data from `data/raw`

In [1]:
import pandas as pd

In [2]:
# COMMENT THE ONE YOU ARE NOT USING OUT

# running locally
RAW_DIR = "../data/raw"
# running on colab
# RAW_DIR = "/content"

In [3]:
pokedex = pd.read_csv(f"{RAW_DIR}/Pokedex_Cleaned.csv", encoding="latin1")
pokedex.head()

,#,Name,Primary Type,Secondary Type,Total,HP,Attack,Defense,Sp.Atk,Sp.Def,Speed,Variant
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,NaN
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,NaN
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,NaN
3,3,Venusaur,Grass,Poison,625,80,100,123,122,120,80,NaN
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,NaN


In [4]:
pokedex_xlsx = pd.read_excel(f"{RAW_DIR}/Pokedex_Cleaned.xlsx")
pokedex_xlsx.head()

,#,Name,Primary Type,Secondary Type,Total,HP,Attack,Defense,Sp.Atk,Sp.Def,Speed,Variant
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,NaN
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,NaN
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,NaN
3,3,Venusaur,Grass,Poison,625,80,100,123,122,120,80,NaN
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,NaN


In [5]:
pokestats = pd.read_csv(f"{RAW_DIR}/PokeStats.csv")
pokestats.head()

,b'# Name Type Total HP Attack Defense Sp. Atk Sp. Def Speed'
0,b'001 BulbasaurGrass Poison\n318\n45\n49\n49\n...
1,b'002 IvysaurGrass Poison\n405\n60\n62\n63\n80...
2,b'003 VenusaurGrass Poison\n525\n80\n82\n83\n1...
3,b'003 Venusaur Mega VenusaurGrass Poison\n625\...
4,b'004 CharmanderFire \n309\n39\n52\n43\n60\n50...


Clean the data (TODO)

In [6]:
pokedex.info()

<class 'pandas.DataFrame'>
RangeIndex: 1173 entries, 0 to 1172
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   #               1173 non-null   int64
 1   Name            1173 non-null   str  
 2   Primary Type    1173 non-null   str  
 3   Secondary Type  633 non-null    str  
 4   Total           1173 non-null   int64
 5   HP              1173 non-null   int64
 6   Attack          1173 non-null   int64
 7   Defense         1173 non-null   int64
 8   Sp.Atk          1173 non-null   int64
 9   Sp.Def          1173 non-null   int64
 10  Speed           1173 non-null   int64
 11  Variant         190 non-null    str  
dtypes: int64(8), str(4)
memory usage: 110.1 KB


In [7]:
pokedex.shape

(1173, 12)

In [8]:
pokedex.isnull().sum()


#                   0
Name                0
Primary Type        0
Secondary Type    540
Total               0
HP                  0
Attack              0
Defense             0
Sp.Atk              0
Sp.Def              0
Speed               0
Variant           983
dtype: int64

In [9]:
pokedex.duplicated().sum()

np.int64(1)

In [10]:
pokestats.info()

<class 'pandas.DataFrame'>
RangeIndex: 1190 entries, 0 to 1189
Data columns (total 1 columns):
 #   Column                                                        Non-Null Count  Dtype
---  ------                                                        --------------  -----
 0   b'# Name Type Total HP Attack Defense Sp. Atk Sp. Def Speed'  1190 non-null   str  
dtypes: str(1)
memory usage: 9.4 KB


In [11]:
pokestats.shape

(1190, 1)

In [12]:
pokestats.isnull().sum()

b'# Name Type Total HP Attack Defense Sp. Atk Sp. Def Speed'    0
dtype: int64

In [13]:
pokestats.duplicated().sum()

np.int64(0)

In [14]:
# Found that pokestats is being read as 1 column

with open(f"{RAW_DIR}/PokeStats.csv", "r", encoding="latin1") as file:
    for i in range(5):
        print(repr(file.readline()))

"b'# Name Type Total HP Attack Defense Sp. Atk Sp. Def Speed'\n"
"b'001 BulbasaurGrass Poison\\n318\\n45\\n49\\n49\\n65\\n65\\n45'\n"
"b'002 IvysaurGrass Poison\\n405\\n60\\n62\\n63\\n80\\n80\\n60'\n"
"b'003 VenusaurGrass Poison\\n525\\n80\\n82\\n83\\n100\\n100\\n80'\n"
"b'003 Venusaur Mega VenusaurGrass Poison\\n625\\n80\\n100\\n123\\n122\\n120\\n80'\n"


In [15]:
import ast
import pandas as pd

type_names = {
    "Normal", "Fire", "Water", "Electric", "Grass", "Ice",
    "Fighting", "Poison", "Ground", "Flying", "Psychic",
    "Bug", "Rock", "Ghost", "Dragon", "Dark", "Steel", "Fairy"
}

rows = []

with open(f"{RAW_DIR}/PokeStats.csv", "r", encoding="latin1") as file:
    for raw_line in file:
        if not raw_line.strip():
            continue

        decoded = ast.literal_eval(
            raw_line.strip()
        ).decode("utf-8", errors="replace")

        parts = decoded.splitlines()

        if parts[0].strip().startswith("#"):
            continue

        first_line = parts[0].strip()
        number = first_line[:3]
        remaining = first_line[3:].strip()
        types = []

        for _ in range(2):
            found_type = None

            for pokemon_type in sorted(
                type_names,
                key=len,
                reverse=True
            ):
                if remaining.endswith(pokemon_type):
                    found_type = pokemon_type
                    remaining = remaining[
                        :-len(pokemon_type)
                    ].strip()
                    break

            if found_type is None:
                break

            types.insert(0, found_type)
        name = remaining
        stats = [
            int(value.strip())
            for value in parts[1:]
            if value.strip()
        ]

        if len(stats) >= 7:
            rows.append([
                number,
                name,
                types[0] if len(types) > 0 else None,
                types[1] if len(types) > 1 else None,
                *stats[:7]
            ])

pokestats = pd.DataFrame(rows, columns=[
    "#", "Name", "Type 1", "Type 2",
    "Total", "HP", "Attack", "Defense",
    "Sp. Atk", "Sp. Def", "Speed"
])

pokestats.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed
0,001,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45
1,002,Ivysaur,Grass,Poison,405,60,62,63,80,80,60
2,003,Venusaur,Grass,Poison,525,80,82,83,100,100,80
3,003,Venusaur Mega Venusaur,Grass,Poison,625,80,100,123,122,120,80
4,004,Charmander,Fire,NaN,309,39,52,43,60,50,65


In [16]:
pokestats.info()

<class 'pandas.DataFrame'>
RangeIndex: 1190 entries, 0 to 1189
Data columns (total 11 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   #        1190 non-null   str  
 1   Name     1190 non-null   str  
 2   Type 1   1190 non-null   str  
 3   Type 2   648 non-null    str  
 4   Total    1190 non-null   int64
 5   HP       1190 non-null   int64
 6   Attack   1190 non-null   int64
 7   Defense  1190 non-null   int64
 8   Sp. Atk  1190 non-null   int64
 9   Sp. Def  1190 non-null   int64
 10  Speed    1190 non-null   int64
dtypes: int64(7), str(4)
memory usage: 102.4 KB


In [17]:
# checking to see if it worked
pokestats.shape

(1190, 11)

In [18]:
# removed duplicate pokemon records
pokedex = pokedex.drop_duplicates().copy()
pokestats = pokestats.drop_duplicates().copy()

#replaced missing "Secondary Types" to None
pokedex["Secondary Type"] = pokedex["Secondary Type"].fillna("None")
#replaced missing "Variants" to None
pokedex["Variant"] = pokedex["Variant"].fillna("None")
#replaced missing "Type 2" to None
pokestats["Type 2"] = pokestats["Type 2"].fillna("None")

stat_columns = [
    "Total", "HP", "Attack", "Defense",
    "Sp. Atk", "Sp. Def", "Speed"
]

for column in stat_columns:
    pokestats[column] = pd.to_numeric(
        pokestats[column], errors="coerce"
    )

In [19]:
pokedex.to_csv(f"{RAW_DIR}/Pokedex_Cleaned_Final.csv", index=False)
pokestats.to_csv(f"{RAW_DIR}/PokeStats_Cleaned_Final.csv", index=False)